# VTI Analysis — <DEVICE>

Analysis notebook template for VTI runs (spxtr/measureme `sweep` data layout:
`data/<i>/data.tsv[.gz]` + `metadata.json`).

`metadata.hera.policy = edit_exec` — the agent may edit/append/execute
cells here. Hardware stays on the measure notebook, operator-run.

In [ ]:
import os
import json

import numpy as np
import scipy.constants as cst
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

from sweep import sweep_load as sl
from pathlib import Path

%matplotlib inline

cst_e = cst.e
cst_h = cst.h
Rq = cst_h/cst_e**2

def _resolve_data_dir():
    """Locate this run's ./data dir from the notebook's own location.

    The ana kernel's cwd is not guaranteed to be the measurement dir, so a
    bare './data' breaks loaders with FileNotFoundError. IPython sets
    globals()['_dh'] to a list whose first entry is the notebook's startup
    cwd (the dir holding the notebook); this survives jupyter-collaboration/
    RTC and any later %cd, so we prefer it. We fall back to Path.cwd() and
    search a few levels up so loading still works if the kernel started inside
    ./data/<id>/. The walk stops before the filesystem root so an unrelated
    top-level 'data' dir can never be matched.
    """
    candidates = []
    dh = globals().get("_dh")
    if dh:
        candidates.append(Path(dh[0]))
    candidates.append(Path.cwd())
    for base in candidates:
        base = Path(base).resolve()
        for d in [base, *base.parents]:
            if d == d.parent:  # filesystem root
                break
            cand = d / "data"
            if cand.is_dir():
                return str(cand)
    return str((Path(candidates[0]) / "data").resolve())


file_path = _resolve_data_dir()


def load1d(i):
    return np.loadtxt(os.path.join(file_path, "{:d}".format(i), "data.tsv.gz"))

def load_meta(i):
    fp = open(os.path.join(file_path, "{:d}".format(i), "metadata.json"), "r")
    return json.load(fp)

def pretty_load(i):
    data = load1d(i)
    meta = load_meta(i)

    data_dict = {}
    for ind, col in enumerate(meta['columns']):
        data_dict[col] = data[:, ind]

    return data_dict

def pretty_load2d(i):
    data = np.loadtxt(os.path.join(file_path, "{:d}".format(i), "data.tsv"))
    meta = load_meta(i)

    data_dict = {}
    for ind, col in enumerate(meta['columns']):
        data_dict[col] = data[:, ind]

    return data_dict

## (n, D) — capacitance-based convention

Density/displacement from areal gate capacitances (`calc_nD`), **not** the
toploader's `dtg/dbg` constant form. `n` comes out in SI (m^-2; multiply by
1e-16 for 10^12 cm^-2); `D` in V/nm.

In [ ]:
cst_epsilon0 = cst.epsilon_0
cst_e = cst.e
cst_h = cst.h
cst_phi0 = cst_h/(2*cst_e)
cst_kappa_SiO2 = 3.9
cst_kappa_hBN = 3

def calc_ctg(t_tg):
    """
    Calculates area capacitance for a top gate
    """
    return cst_epsilon0*cst_kappa_hBN/t_tg

def calc_cbg(t_bg, t_bgSiO2):
    """
    Calculates area capacitance for a back gate
    For a metal back gate, set t_bgSiO2 = 0
    """
    return cst_epsilon0*1/(t_bg/cst_kappa_hBN + t_bgSiO2/cst_kappa_SiO2)

def calc_nD(V_tg, V_bg, c_tg, c_bg, n_offset=0, D_offset = 0):
    n = (1/cst_e) * (c_tg*V_tg  + c_bg*V_bg) + n_offset
    D = 0.5*(c_tg*V_tg - c_bg*V_bg)*1/cst_epsilon0 + D_offset
    return n, D


# Device gate-stack constants — fill in per device (meters)
t_tg = <T_TG>          # top-gate hBN thickness
t_bg = <T_BG>          # back-gate hBN thickness
t_bgSiO2 = <T_BGSIO2>  # back-gate SiO2 thickness (0 for metal/graphite back gate)

c_tg = calc_ctg(t_tg)
c_bg = calc_cbg(t_bg, t_bgSiO2)

## Fan re-plot pattern

On this rig the current channel is **`sr0` read through the Ithaco preamp**:
the explicit gain division `I = sr0_X / measurement_config['sr0_gain']` is
**mandatory** — skipping it inverts/rescales every R. (`sr0_gain` is negative,
e.g. -1e6.) The on-disk `plot.png` is only the live monitor; always re-form
R from the TSV.

In [ ]:
# PATTERN — Landau-fan re-plot (set the file id).
dat = sl.pload(file_path, <FILE_ID>)

n, D = calc_nD(dat['xs'][0], dat['xs'][1], c_tg, c_bg)
n_12cm = n * 1e-16  # 10^12 cm^-2
ys = dat['ys'][0]   # B (T)

# Current — explicit gain division is MANDATORY on this rig
curr = dat['sr0_X'] / dat['measurement_config']['sr0_gain']

r_xx = ['sr1_X', 'sr2_X']
r_xy = ['sr3_X']

nu = <NU>
nn = cst.e * ys * nu / cst.h * 1e-16  # filling-factor line n = nu·e·B/h, in 10^12 cm^-2

norm = mcolors.SymLogNorm(linthresh=50, vmin=0, vmax=100e3)
for r_xx_pair in r_xx:
    data = dat[r_xx_pair] / curr
    plt.figure()
    plt.pcolormesh(n_12cm, ys, data, cmap='viridis', norm=norm)
    plt.colorbar(label='$R_{xx}$ (Ohms)')
    plt.xlabel('$n$ 10$^{12}$ (cm$^{-2}$)')
    plt.ylabel('$B$ (T)')

h_e2 = cst.h/cst.e**2
for r_xy_pair in r_xy:
    data = dat[r_xy_pair] / curr
    plt.figure()
    plt.pcolormesh(n_12cm, ys, data, cmap='seismic',
                   vmax=h_e2/nu, vmin=-h_e2/nu)
    plt.colorbar(label='$R_{xy}$ (Ohms)')
    plt.xlabel('$n$ 10$^{12}$ (cm$^{-2}$)')
    plt.ylabel('$B$ (T)')
    plt.plot(nn, ys, '--')

plt.figure(dpi=240)
plt.pcolormesh(n_12cm, ys, curr*1e9, cmap='inferno')
plt.colorbar(label='I (nA)')
plt.xlabel('$n$ 10$^{12}$ (cm$^{-2}$)')
plt.ylabel('$B$ (T)')

## Filling-factor overlay pattern

In [ ]:
# PATTERN — filling-factor overlay: nu lines n = nu·e·B/h on the current fan
# axes (n axis in 10^12 cm^-2 → factor 1e-16 from SI m^-2).
for nu in [<NU_LIST>]:
    nn = cst.e * ys * nu / cst.h * 1e-16
    plt.plot(nn, ys, '--', lw=1, label=rf'$\nu = {nu}$')
plt.legend()